# Business Cases with the ML App Python client

A compact, rerunnable tour of the typed Business Case API. It creates or reuses one example BC and never performs permanent deletion.

In [2]:
from pathlib import Path
import sys

repository_root = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'ml_app_client').is_dir()), None)
if repository_root is None:
    raise RuntimeError('Start Jupyter inside the ml-app repository')
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from ml_app_client import MLAppClient, ResourceNotFoundError

client = MLAppClient.connect()
print('Connected as', client.me().get('login_name'))

Connected as test3@example.pl


## Create once, then reuse

Names are globally unique. `ensure_business_case` is safe for a notebook re-run and reports whether this execution created the BC.

In [3]:
CASE_NAME = '[MLAPP client module] Customer churn demo'

case, created = client.ensure_business_case(
    name=CASE_NAME,
    description='Small, rerunnable client-module example.',
    problem_type='binary_classification',
    business_goal='Reduce voluntary customer churn.',
    primary_metric='roc_auc',
    target_column='churned',
    success_criteria='ROC-AUC of at least 0.82.',
)
print('CREATED' if created else 'REUSED', case.id, case.name, case.status)

REUSED 413a4d16-235f-48d7-957b-cc02592e7ef0 [MLAPP client module] Customer churn demo draft


In [3]:
case

BusinessCase(id='413a4d16-235f-48d7-957b-cc02592e7ef0', name='[MLAPP client module] Customer churn demo', description='Small, rerunnable client-module example.', problem_type='binary_classification', status='draft', owner_id='c3945744-aa0b-4d94-8c84-24a7d46fbe8c', access_role='owner', business_owner='', primary_metric='roc_auc', target_column='churned', business_goal='Reduce voluntary customer churn.', success_criteria='ROC-AUC of at least 0.82.', raw={'id': '413a4d16-235f-48d7-957b-cc02592e7ef0', 'owner_id': 'c3945744-aa0b-4d94-8c84-24a7d46fbe8c', 'name': '[MLAPP client module] Customer churn demo', 'description': 'Small, rerunnable client-module example.', 'problem_type': 'binary_classification', 'status': 'draft', 'business_owner': '', 'primary_metric': 'roc_auc', 'target_column': 'churned', 'business_goal': 'Reduce voluntary customer churn.', 'success_criteria': 'ROC-AUC of at least 0.82.', 'created_by': 'c3945744-aa0b-4d94-8c84-24a7d46fbe8c', 'updated_by': 'c3945744-aa0b-4d94-8c84

## Read, update, and search

Models expose attributes, but remain compatible with legacy mapping access such as `case['id']`. Pages are bounded and report the exact filtered total.

In [6]:
case = client.get_business_case_by_name(CASE_NAME)
display(case)
display(case.status)

BusinessCase(id='413a4d16-235f-48d7-957b-cc02592e7ef0', name='[MLAPP client module] Customer churn demo', description='Small, rerunnable client-module example.', problem_type='binary_classification', status='draft', owner_id='c3945744-aa0b-4d94-8c84-24a7d46fbe8c', access_role='owner', business_owner='', primary_metric='roc_auc', target_column='churned', business_goal='Reduce voluntary customer churn.', success_criteria='ROC-AUC of at least 0.82.', raw=BusinessCase(id='413a4d16-235f-48d7-957b-cc02592e7ef0', name='[MLAPP client module] Customer churn demo', description='Small, rerunnable client-module example.', problem_type='binary_classification', status='draft', owner_id='c3945744-aa0b-4d94-8c84-24a7d46fbe8c', access_role='owner', business_owner='', primary_metric='roc_auc', target_column='churned', business_goal='Reduce voluntary customer churn.', success_criteria='ROC-AUC of at least 0.82.', raw={'id': '413a4d16-235f-48d7-957b-cc02592e7ef0', 'owner_id': 'c3945744-aa0b-4d94-8c84-24a7

'draft'

In [7]:
case = client.update_business_case(case, status='active')
display(case.status)

'active'

In [9]:
display(case.description)
case = client.update_business_case(case, description="Just a test")
display(case.description)

'Small, rerunnable client-module example.'

'Just a test'

In [10]:
page = client.page_business_cases(search='churn', limit=10)
print(f'{page.total} matching BC(s); current status: {case.status}')
[(item.id, item.name, item.access_role) for item in page.items]

1 matching BC(s); current status: active


[('413a4d16-235f-48d7-957b-cc02592e7ef0',
  '[MLAPP client module] Customer churn demo',
  'owner')]

In [11]:
directory = client.page_business_case_catalog(search="churn")
display(directory)

CatalogPage(items=(BusinessCaseCatalogEntry(id='413a4d16-235f-48d7-957b-cc02592e7ef0', name='[MLAPP client module] Customer churn demo', status='active', access_role='owner', request_status='', raw={'id': '413a4d16-235f-48d7-957b-cc02592e7ef0', 'name': '[MLAPP client module] Customer churn demo', 'status': 'active', 'access_role': 'owner', 'request_status': ''}), BusinessCaseCatalogEntry(id='1960dd86-569a-4f63-bb8e-949a6b547e31', name='Storage Subscription Churn', status='draft', access_role='', request_status='', raw={'id': '1960dd86-569a-4f63-bb8e-949a6b547e31', 'name': 'Storage Subscription Churn', 'status': 'draft', 'access_role': '', 'request_status': ''})), total=2, limit=30, offset=0, has_next=False)

## Discover and request access

The organization catalog intentionally exposes minimal information. Use it to find a case and request a role; it does not reveal hidden case details.

In [ ]:
directory = client.page_business_case_catalog(search='churn', limit=10)
[(entry.id, entry.name, entry.access_role, entry.request_status) for entry in directory.items]

# For an inaccessible entry, submit a request explicitly:
# client.request_business_case_access(entry.id, requested_role='reader',
#     justification='I maintain the monthly churn report.')

## Archive instead of deleting

Business Case lineage is retained. Archive it when the work is finished; owners may later reactivate it.

In [ ]:
# client.archive_business_case(case)
# client.activate_business_case(case)
print('No lifecycle change made by this final demonstration cell.')